# 🗑️ LitterVision — Robust Waste Image Classifier

## Introduction

**LitterVision** is an image classification system designed to identify waste into **6 categories**:
`cardboard`, `glass`, `metal`, `paper`, `plastic`, and `trash`.

### Problem Statement
Prior model versions suffered from:
- **Overfitting** on training data
- **Domain shift** — poor performance on real-world, out-of-distribution images
- **Class imbalance** — the `trash` class has fewer samples

### Solution Approach
This notebook builds a robust and generalizable classifier by:
1. Applying aggressive, realistic **data augmentation**
2. Using **MobileNetV2** (ImageNet pretrained) with two-stage fine-tuning
3. Computing and applying **class weights** during training
4. Using **EarlyStopping** and **ReduceLROnPlateau** to prevent overfitting
5. Adding **confidence-based filtering** (`< 0.60 → Unknown`) for safe deployment

---
## 1. Imports & Configuration

In [ ]:
import os
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import Counter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore")

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Global Configuration ─────────────────────────────────────────────────────
DATASET_PATH   = "Dataset"        # Root folder with one sub-folder per class
MODEL_SAVE_PATH = "model.h5"      # Output model path
IMG_SIZE       = 224              # MobileNetV2 native input resolution
BATCH_SIZE     = 32
EPOCHS_FROZEN  = 15               # Stage 1: head only
EPOCHS_FINETUNE = 20              # Stage 2: top layers unfrozen
FINE_TUNE_AT   = 100              # Unfreeze layers from this index onwards
CONFIDENCE_THRESHOLD = 0.60       # Predictions below this → "Unknown"
CLASS_NAMES    = sorted(os.listdir(DATASET_PATH))  # Alphabetically sorted

print(f"TensorFlow version : {tf.__version__}")
print(f"Classes detected   : {CLASS_NAMES}")
print(f"GPU available      : {bool(tf.config.list_physical_devices('GPU'))}")

---
## 2. Data Loading

In [ ]:
def count_images(dataset_path: str) -> dict:
    """Walk dataset folder and return {class_name: image_count} dict."""
    counts = {}
    for cls in sorted(os.listdir(dataset_path)):
        cls_path = os.path.join(dataset_path, cls)
        if os.path.isdir(cls_path):
            images = [
                f for f in os.listdir(cls_path)
                if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
            ]
            counts[cls] = len(images)
    return counts


class_counts = count_images(DATASET_PATH)
total_images = sum(class_counts.values())

print("📂 Dataset Summary")
print("-" * 35)
for cls, cnt in class_counts.items():
    pct = cnt / total_images * 100
    print(f"  {cls:<12} : {cnt:>4} images  ({pct:.1f}%)")
print("-" * 35)
print(f"  {'TOTAL':<12} : {total_images:>4} images")

---
## 3. Data Exploration — Class Distribution

In [ ]:
palette = ["#4CAF50", "#2196F3", "#FF5722", "#9C27B0", "#FF9800", "#00BCD4"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(
    class_counts.keys(),
    class_counts.values(),
    color=palette,
    edgecolor="white",
    linewidth=1.5,
    zorder=3
)

# Annotate counts
for bar, count in zip(bars, class_counts.values()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 3,
        str(count),
        ha="center", va="bottom",
        fontsize=12, fontweight="bold", color="#333"
    )

ax.set_title("LitterVision — Class Distribution", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("Waste Category", fontsize=13)
ax.set_ylabel("Number of Images", fontsize=13)
ax.set_facecolor("#f9f9f9")
fig.patch.set_facecolor("#f9f9f9")
ax.grid(axis="y", linestyle="--", alpha=0.5, zorder=1)
ax.spines[["top", "right"]].set_visible(False)

# Highlight smallest class
min_cls  = min(class_counts, key=class_counts.get)
min_idx  = list(class_counts.keys()).index(min_cls)
bars[min_idx].set_edgecolor("red")
bars[min_idx].set_linewidth(2.5)
ax.annotate(
    "⚠ Smallest class",
    xy=(bars[min_idx].get_x() + bars[min_idx].get_width() / 2,
        class_counts[min_cls] + 10),
    xytext=(bars[min_idx].get_x() + 1.0, class_counts[min_cls] + 60),
    arrowprops=dict(arrowstyle="->", color="red"),
    color="red", fontsize=10
)

plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4. Preprocessing — Image Generators

In [ ]:
# ── Augmented training generator ────────────────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.20,
    # Geometric transforms
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.20,
    horizontal_flip=True,
    vertical_flip=False,
    shear_range=0.10,
    # Color / photometric
    brightness_range=[0.70, 1.30],
    channel_shift_range=20.0,
    fill_mode="nearest"
)

# ── Validation generator (NO augmentation — only rescaling) ─────────────────
val_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.20
)

train_gen = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True,
    seed=SEED
)

val_gen = val_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
    seed=SEED
)

# Class-index mapping aligned with generators
CLASS_INDICES = train_gen.class_indices
IDX_TO_CLASS  = {v: k for k, v in CLASS_INDICES.items()}

print("Class → Index mapping:", CLASS_INDICES)
print(f"Training batches   : {len(train_gen)}")
print(f"Validation batches : {len(val_gen)}")

---
## 5. Augmentation — Visual Sanity Check

In [ ]:
def show_augmented_samples(dataset_path: str, class_indices: dict,
                           datagen: ImageDataGenerator,
                           n_classes: int = 6, n_variants: int = 4) -> None:
    """
    Display augmented variants for one image from each class.
    Rows = classes | Columns = augmented variants
    """
    fig, axes = plt.subplots(n_classes, n_variants + 1,
                             figsize=(3 * (n_variants + 1), 3 * n_classes))
    fig.suptitle("Augmented Samples per Class", fontsize=16, fontweight="bold", y=1.01)

    aug_gen = ImageDataGenerator(
        rotation_range=30,
        width_shift_range=0.15,
        height_shift_range=0.15,
        zoom_range=0.20,
        horizontal_flip=True,
        brightness_range=[0.70, 1.30],
        channel_shift_range=20.0,
        fill_mode="nearest"
    )

    for row, cls_name in enumerate(sorted(class_indices.keys())):
        cls_dir = os.path.join(dataset_path, cls_name)
        img_file = random.choice([
            f for f in os.listdir(cls_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])
        img_path = os.path.join(cls_dir, img_file)
        orig_img = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
        orig_arr = img_to_array(orig_img).astype("uint8")

        # Column 0: original
        axes[row, 0].imshow(orig_arr)
        axes[row, 0].set_title("Original", fontsize=9)
        axes[row, 0].set_ylabel(cls_name.capitalize(), fontsize=11, fontweight="bold")

        # Columns 1-n: augmented
        arr_4d = np.expand_dims(img_to_array(orig_img), axis=0)
        aug_iter = aug_gen.flow(arr_4d, batch_size=1, seed=SEED)
        for col in range(1, n_variants + 1):
            aug_img = next(aug_iter)[0].astype("uint8")
            axes[row, col].imshow(aug_img)
            axes[row, col].set_title(f"Aug {col}", fontsize=9)

        for ax in axes[row]:
            ax.axis("off")

    plt.tight_layout()
    plt.savefig("augmentation_samples.png", dpi=120, bbox_inches="tight")
    plt.show()


show_augmented_samples(DATASET_PATH, CLASS_INDICES, train_datagen)

---
## 6. Class Imbalance — Computing Class Weights

In [ ]:
# Reconstruct label array from generator file list
train_labels = train_gen.classes  # integer class indices for every training sample

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)

class_weights = dict(enumerate(class_weights_array))

print("⚖️  Class Weights (higher = penalised more during training)")
print("-" * 45)
for idx, weight in class_weights.items():
    print(f"  [{idx}] {IDX_TO_CLASS[idx]:<12} → weight = {weight:.4f}")

---
## 7. Model Building — MobileNetV2 + Custom Head

In [ ]:
def build_model(num_classes: int, img_size: int = 224) -> Model:
    """
    Build a MobileNetV2-based transfer-learning classifier.

    Architecture:
        MobileNetV2 (frozen) → GlobalAvgPool → Dropout → Dense(256) →
        BatchNorm → Dropout → Dense(num_classes, softmax)
    """
    base = MobileNetV2(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights="imagenet"
    )
    base.trainable = False   # Stage 1: freeze entire base

    inputs = keras.Input(shape=(img_size, img_size, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.40)(x)
    x = layers.Dense(256, activation="relu",
                      kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs, name="LitterVision_MobileNetV2")
    return model, base


NUM_CLASSES = len(CLASS_INDICES)
model, base_model = build_model(NUM_CLASSES, IMG_SIZE)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

---
## 8. Training

### Stage 1 — Train Classification Head (base frozen)

In [ ]:
# ── Callbacks ────────────────────────────────────────────────────────────────
early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath="best_model_stage1.h5",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

# ── Stage 1 fit ───────────────────────────────────────────────────────────────
print("🚀 Stage 1: Training classification head (base frozen)")
history_s1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FROZEN,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

print(f"\n✅ Stage 1 complete — best val_accuracy: "
      f"{max(history_s1.history['val_accuracy']):.4f}")

### Stage 2 — Fine-Tuning (top layers unfrozen)

In [ ]:
# Unfreeze layers from FINE_TUNE_AT index onwards
base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f"Unfrozen layers in base: {trainable_count} / {len(base_model.layers)}")

# Recompile with lower LR for fine-tuning
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

checkpoint_ft = ModelCheckpoint(
    filepath="best_model_finetuned.h5",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

print("\n🔥 Stage 2: Fine-tuning top MobileNetV2 layers")
history_s2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FINETUNE,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr, checkpoint_ft],
    verbose=1
)

print(f"\n✅ Stage 2 complete — best val_accuracy: "
      f"{max(history_s2.history['val_accuracy']):.4f}")

### Training Curves

In [ ]:
def merge_histories(h1, h2):
    """Concatenate two Keras history dicts for combined plotting."""
    combined = {}
    for key in h1.history:
        combined[key] = h1.history[key] + h2.history.get(key, [])
    return combined


def plot_training_curves(history_dict: dict, stage_boundary: int) -> None:
    """Plot accuracy and loss curves with stage boundary annotation."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(history_dict["accuracy"]) + 1)

    # ── Accuracy ─────────────────────────────────────────────────────────────
    ax1.plot(epochs, history_dict["accuracy"],     label="Train Acc",  lw=2, color="#2196F3")
    ax1.plot(epochs, history_dict["val_accuracy"], label="Val Acc",    lw=2, color="#4CAF50", linestyle="--")
    ax1.axvline(stage_boundary, color="#FF5722", linestyle=":", lw=1.5, label="Fine-tune start")
    ax1.fill_between(epochs, history_dict["accuracy"], history_dict["val_accuracy"],
                     alpha=0.08, color="gray")
    ax1.set_title("Accuracy", fontsize=14, fontweight="bold")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy")
    ax1.legend(); ax1.grid(alpha=0.3)
    ax1.set_ylim(0, 1.05)

    # ── Loss ─────────────────────────────────────────────────────────────────
    ax2.plot(epochs, history_dict["loss"],     label="Train Loss", lw=2, color="#2196F3")
    ax2.plot(epochs, history_dict["val_loss"], label="Val Loss",   lw=2, color="#F44336", linestyle="--")
    ax2.axvline(stage_boundary, color="#FF5722", linestyle=":", lw=1.5, label="Fine-tune start")
    ax2.set_title("Loss", fontsize=14, fontweight="bold")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
    ax2.legend(); ax2.grid(alpha=0.3)

    fig.suptitle("LitterVision — Training Curves", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()


combined_history = merge_histories(history_s1, history_s2)
stage_boundary   = len(history_s1.history["accuracy"])
plot_training_curves(combined_history, stage_boundary)

---
## 9. Save Final Model

In [ ]:
model.save(MODEL_SAVE_PATH)
print(f"✅ Final model saved → {MODEL_SAVE_PATH}")

# Print final metrics
final_train_acc = combined_history["accuracy"][-1]
final_val_acc   = combined_history["val_accuracy"][-1]
best_val_acc    = max(combined_history["val_accuracy"])
print(f"   Last  train accuracy : {final_train_acc:.4f}")
print(f"   Last  val   accuracy : {final_val_acc:.4f}")
print(f"   Best  val   accuracy : {best_val_acc:.4f}")

---
## 10. Evaluation

### Confusion Matrix & Classification Report

In [ ]:
# Load best fine-tuned checkpoint for evaluation
best_model = keras.models.load_model("best_model_finetuned.h5")

# Reset generator to start from beginning
val_gen.reset()

y_pred_probs = best_model.predict(val_gen, verbose=1)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = val_gen.classes

label_names  = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]

print("\n📊 Classification Report")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=label_names))

In [ ]:
def plot_confusion_matrix(y_true, y_pred, label_names):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)  # Row-normalised

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for ax, data, title, fmt in zip(
        axes,
        [cm, cm_norm],
        ["Confusion Matrix (Counts)", "Confusion Matrix (Normalised)"],
        ["d", ".2f"]
    ):
        sns.heatmap(
            data, annot=True, fmt=fmt, cmap="Blues",
            xticklabels=label_names, yticklabels=label_names,
            linewidths=0.5, linecolor="white", ax=ax
        )
        ax.set_title(title, fontsize=13, fontweight="bold", pad=12)
        ax.set_xlabel("Predicted Label", fontsize=11)
        ax.set_ylabel("True Label",      fontsize=11)
        ax.tick_params(axis="x", rotation=30)

    plt.suptitle("LitterVision — Evaluation Results", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()


plot_confusion_matrix(y_true, y_pred, label_names)

---
## 11. Testing on External Images

Place any real-world image into the `test_images/` folder and run the cell below.  
Predictions with confidence **below 0.60** are flagged as `"Unknown"` to avoid overconfident wrong answers.

In [ ]:
def predict_single_image(
    image_path: str,
    model: Model,
    idx_to_class: dict,
    img_size: int = 224,
    threshold: float = 0.60
) -> dict:
    """
    Predict waste category for a single image.

    Returns:
        dict with keys: image_path, predicted_class, confidence, raw_probs, all_scores
    """
    img = load_img(image_path, target_size=(img_size, img_size))
    arr = img_to_array(img) / 255.0
    arr = np.expand_dims(arr, axis=0)   # (1, H, W, 3)

    probs     = model.predict(arr, verbose=0)[0]
    pred_idx  = int(np.argmax(probs))
    confidence = float(probs[pred_idx])

    predicted_class = idx_to_class[pred_idx] if confidence >= threshold else "Unknown"

    return {
        "image_path":      image_path,
        "predicted_class": predicted_class,
        "confidence":      confidence,
        "raw_probs":       probs,
        "all_scores":      {idx_to_class[i]: float(probs[i]) for i in range(len(probs))}
    }


def display_prediction(result: dict) -> None:
    """Display image alongside its prediction and per-class probability bar chart."""
    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(12, 4),
                                          gridspec_kw={"width_ratios": [1, 2]})

    # Image panel
    img = load_img(result["image_path"], target_size=(IMG_SIZE, IMG_SIZE))
    ax_img.imshow(img)
    ax_img.axis("off")

    pred   = result["predicted_class"]
    conf   = result["confidence"]
    color  = "#4CAF50" if pred != "Unknown" else "#FF5722"
    ax_img.set_title(
        f"{'✅' if pred != 'Unknown' else '❓'} {pred.upper()}\n"
        f"Confidence: {conf:.1%}",
        fontsize=13, fontweight="bold", color=color
    )

    # Probability bar chart
    scores = result["all_scores"]
    bars = ax_bar.barh(
        list(scores.keys()),
        list(scores.values()),
        color=[color if k == result["predicted_class"] else "#BDBDBD"
               for k in scores],
        edgecolor="white"
    )
    ax_bar.axvline(CONFIDENCE_THRESHOLD, color="#FF5722",
                   linestyle="--", lw=1.5, label=f"Threshold ({CONFIDENCE_THRESHOLD:.0%})")
    ax_bar.set_xlim(0, 1)
    ax_bar.set_xlabel("Probability", fontsize=11)
    ax_bar.set_title("Per-class Scores", fontsize=12, fontweight="bold")
    ax_bar.legend(fontsize=9)

    for bar, val in zip(bars, scores.values()):
        ax_bar.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
                    f"{val:.1%}", va="center", fontsize=9)

    plt.tight_layout()
    plt.show()
    print("-" * 50)


print("Prediction functions defined. Run the next cell to test external images.")

In [ ]:
# ── Test on external images ───────────────────────────────────────────────────
# Place your real-world images in the `test_images/` folder, or update paths below.

TEST_IMAGE_DIR = "test_images"
os.makedirs(TEST_IMAGE_DIR, exist_ok=True)

test_image_paths = [
    p for p in Path(TEST_IMAGE_DIR).glob("*")
    if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp")
]

if not test_image_paths:
    print("⚠️  No images found in 'test_images/' folder.")
    print("   Add some real-world waste images and re-run this cell.")
else:
    print(f"Found {len(test_image_paths)} image(s). Running predictions...\n")
    for img_path in test_image_paths:
        result = predict_single_image(
            str(img_path), best_model, IDX_TO_CLASS,
            img_size=IMG_SIZE, threshold=CONFIDENCE_THRESHOLD
        )
        display_prediction(result)

#### — Quick single-image prediction (no folder needed) —

In [ ]:
# Change this path to any image on your machine
# SINGLE_IMAGE_PATH = "path/to/your/image.jpg"

# result = predict_single_image(
#     SINGLE_IMAGE_PATH, best_model, IDX_TO_CLASS,
#     img_size=IMG_SIZE, threshold=CONFIDENCE_THRESHOLD
# )
# display_prediction(result)

---
## 12. Bonus — Research Improvement Suggestions

The current model performs well on held-out validation data. Below are **two research-grade directions** that could further boost real-world robustness:

---

### 💡 Improvement 1 — Domain Generalization via AugMix / RandAugment

**Problem**: Domain shift occurs when test images differ from training images in lighting, background, camera angle, or JPEG compression artifacts.

**Solution**: Apply **AugMix** (Hendrycks et al., 2020), which mixes multiple augmentation chains and encourages the model to learn invariant features regardless of image style.

```python
# Pseudocode — integrates with tf.data pipeline
# pip install augly  OR  use TensorFlow Extended (tfx) augmentation
import tensorflow_addons as tfa

def augmix_pipeline(image):
    ops = [
        lambda x: tfa.image.rotate(x, angles=tf.random.uniform([], -0.4, 0.4)),
        lambda x: tf.image.random_contrast(x, 0.6, 1.4),
        lambda x: tfa.image.sharpness(x, 0.5)
    ]
    k = tf.random.uniform([], 1, 4, dtype=tf.int32)
    selected = tf.random.shuffle(ops)[:k]
    for op in selected:
        image = op(image)
    return image
```

**Expected impact**: +3–7% accuracy on out-of-distribution images without any new labelled data.

---

### 💡 Improvement 2 — Uncertainty Quantification via Monte Carlo Dropout

**Problem**: Standard softmax confidence scores are overconfident — a model can output 95% confidence on a completely wrong class.

**Solution**: Use **MC Dropout** (Gal & Ghahramani, 2016) — keep Dropout **active at inference** and run `T` forward passes. The variance across passes measures epistemic uncertainty.

```python
def mc_predict(model, image_array, T=30):
    """T stochastic forward passes with Dropout active."""
    preds = np.stack([
        model(image_array, training=True)  # training=True keeps Dropout ON
        for _ in range(T)
    ])  # shape: (T, 1, num_classes)
    mean_pred  = preds.mean(axis=0)   # Mean prediction
    uncertainty = preds.std(axis=0)   # Std = epistemic uncertainty
    return mean_pred, uncertainty
```

**Expected impact**: Calibrated uncertainties enable the model to say *"I don't know"* with principled evidence, reducing silent misclassification in production.

---
**Combined approach**: Using AugMix for training diversity AND MC Dropout for uncertainty-aware inference would yield a production-grade, safety-conscious classifier.

---
## 13. Conclusion

| Aspect | Approach Used |
|---|---|
| **Base Model** | MobileNetV2 (ImageNet pretrained) |
| **Training Strategy** | 2-stage: frozen head → fine-tune top layers |
| **Augmentation** | Rotation, shift, zoom, brightness, channel shift |
| **Class Imbalance** | Balanced class weights via `sklearn` |
| **Regularization** | Dropout (0.4 + 0.3), L2, BatchNorm |
| **Callbacks** | EarlyStopping, ReduceLROnPlateau, ModelCheckpoint |
| **Confidence Filter** | Threshold at 0.60 → Unknown |
| **Evaluation** | Confusion matrix, precision/recall/F1 per class |

The final model is saved to `model.h5` and ready to be loaded by the Flask backend (`app.py`).

For deployment, replace the `model.h5` in your project root with `best_model_finetuned.h5` (the checkpoint with the best validation accuracy).